In [ ]:
import os as _ka_os2
WORKING_DIR = "/kaggle/working" if _ka_os2.path.exists("/kaggle/working") else _ka_os2.getcwd()
SCRATCH_DIR = "/kaggle/tmp" if _ka_os2.path.exists("/kaggle/tmp") else _ka_os2.getcwd()


In [ ]:
import sys as _ka_sys
try:
    _ka_sys.stdout.reconfigure(line_buffering=True)
    _ka_sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass


In [ ]:
%%capture
!uv pip install --system sentencepiece protobuf huggingface_hub hf_transfer
!uv pip install --system --upgrade --no-deps accelerate bitsandbytes
!uv pip install --system --upgrade --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
import torch; print(torch.__version__, torch.cuda.device_count(), flush=True)


In [ ]:
import os, time, urllib.request
os.makedirs(SCRATCH_DIR, exist_ok=True)
os.makedirs(WORKING_DIR, exist_ok=True)
_url = "https://github.com/barryallen16/sivabharani-comments/raw/refs/heads/main/processing_notebook/eval_500.jsonl"
_dst = SCRATCH_DIR + "/eval_500.jsonl"
for _t in range(5):
    try:
        urllib.request.urlretrieve(_url, _dst)
        break
    except Exception as _e:
        print(f"download try {_t + 1} failed: {_e}", flush=True)
        time.sleep(15)
else:
    raise RuntimeError("eval_500.jsonl download failed after retries")
print(f"downloaded {_dst}", flush=True)


In [ ]:
import torch
print(f"gpus={torch.cuda.device_count()}", flush=True)
assert torch.cuda.device_count() > 0, "need T4 x2"
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
ner = pipeline("token-classification", model="ai4bharat/IndicNER", aggregation_strategy="simple", device=0, batch_size=128)
def ner_has_person(text):
    try:
        ents = ner(text)
    except Exception:
        return True
    return any("PER" in e.get("entity_group", "") for e in ents)
bnb = BitsAndBytesConfig(load_in_4bit=True)
tokenizer = AutoTokenizer.from_pretrained("Tamil-ai/tamil-qwen25-7b-instruct")
model = AutoModelForCausalLM.from_pretrained("Tamil-ai/tamil-qwen25-7b-instruct", quantization_config=bnb, device_map="auto")
print("models loaded", flush=True)


In [ ]:
import json, time
SYS = ('Extract person names from Tamil text. Reply JSON only: {"tamil": [...], "english": [...]}. Empty lists if none.\n'
'Rules: names must be Tamil-script person names only. Never output English words, brands, objects, greetings (bro, hello, hey, you, iphone, processor).\n'
'English must list the transliteration of each Tamil name in the same order - never empty when tamil is non-empty.\n'
'Only extract names that appear verbatim in the Input. Never copy names from these examples.\n'
'Example: Input: வணக்கம், நான் சூர்யா. Output: {"tamil": ["சூர்யா"], "english": ["Surya"]}\n'
'Example: Input: ப்ரோ iphone processor நல்லா இருக்கு. Output: {"tamil": [], "english": []}\n'
'Input: {TEXT}\nOutput:')
def llm_extract(text):
    prompt = SYS.replace("{TEXT}", text)
    msgs = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    try:
        _dev = model.model.embed_tokens.weight.device
    except Exception:
        _dev = "cuda:0"
    inputs = inputs.to(_dev)
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    cleaned = gen.replace("json", "").replace("```", "").strip()
    try:
        return json.loads(cleaned), False
    except Exception:
        return {"tamil": [], "english": []}, True
def has_names(ex):
    return bool(ex.get("tamil")) or bool(ex.get("english"))
rows = [json.loads(l) for l in open(SCRATCH_DIR + "/eval_500.jsonl", encoding="utf-8") if l.strip()]
print(f"eval rows={len(rows)}", flush=True)
assert len(rows) == 500, f"eval file broken: {len(rows)} rows"
t0 = time.perf_counter()
ner_flags = [ner_has_person(r["text"]) for r in rows]
ner_secs = time.perf_counter() - t0
print(f"NER pass: {ner_secs:.1f}s", flush=True)
preds = []
t1 = time.perf_counter()
llm_calls = 0
fails = 0
for r, flag in zip(rows, ner_flags):
    if flag:
        ph, f = llm_extract(r["text"])
        llm_calls += 1
        fails += f
    else:
        ph = {"tamil": [], "english": []}
    pn = {"tamil": ["?"], "english": ["?"]} if flag else {"tamil": [], "english": []}
    preds.append((pn, ph))
hybrid_secs = time.perf_counter() - t1
print(f"hybrid LLM: {hybrid_secs:.1f}s calls={llm_calls} fails={fails}", flush=True)
t2 = time.perf_counter()
preds_all = []
fails_all = 0
for ri, r in enumerate(rows):
    p, f = llm_extract(r["text"])
    preds_all.append(p)
    fails_all += f
    if (ri + 1) % 100 == 0:
        print(f"llm-only {ri + 1}/{len(rows)}", flush=True)
llm_secs = time.perf_counter() - t2
vram_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"llm-only: {llm_secs:.1f}s fails={fails_all} vram={vram_gb:.1f}GB", flush=True)


In [ ]:
def score(pairs):
    tp = fp = fn = tem = een = 0
    for ref, p in pairs:
        rh, ph = has_names(ref), has_names(p)
        tp += rh and ph
        fp += (not rh) and ph
        fn += rh and (not ph)
        if rh and ph:
            tem += p.get("tamil") == ref.get("tamil")
            een += p.get("english") == ref.get("english")
    pr = tp / (tp + fp) if tp + fp else 0.0
    rc = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
    return {"P": round(pr, 3), "R": round(rc, 3), "F1": round(f1, 3), "tamil_exact": tem, "eng_exact": een}
m = {
    "ner": score([(r["ref_extract"], pn) for r, (pn, ph) in zip(rows, preds)]),
    "hybrid": score([(r["ref_extract"], ph) for r, (pn, ph) in zip(rows, preds)]),
    "llm_only": score([(r["ref_extract"], p) for r, p in zip(rows, preds_all)]),
}
m["model"] = "Tamil-ai/tamil-qwen25-7b-instruct"
m["time"] = {"ner_secs": round(ner_secs, 1), "hybrid_llm_secs": round(hybrid_secs, 1), "llm_all_secs": round(llm_secs, 1), "llm_calls_hybrid": llm_calls, "fails_hybrid": fails, "fails_llm": fails_all, "vram_gb": round(vram_gb, 2)}
print(json.dumps(m, indent=1), flush=True)
with open(WORKING_DIR + "/eval_metrics_qwen.json", "w", encoding="utf-8") as f:
    json.dump(m, f, ensure_ascii=False, indent=1)
with open(WORKING_DIR + "/eval_preds_qwen.jsonl", "w", encoding="utf-8") as f:
    for r, (pn, ph), pa in zip(rows, preds, preds_all):
        json.dump({"video_id": r["video_id"], "text": r["text"], "ref": r["ref_extract"], "ner": pn, "hybrid": ph, "llm": pa}, f, ensure_ascii=False)
        f.write("\n")
print("saved eval_metrics_qwen.json + eval_preds_qwen.jsonl", flush=True)


In [ ]:
import gc, shutil, os
for _n in ["model", "tokenizer", "ner"]:
    try:
        del globals()[_n]
    except KeyError:
        pass
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass
for _p in [os.path.expanduser("~/.cache/huggingface"), os.path.expanduser("~/.cache/uv"), "/root/.cache/huggingface"]:
    try:
        if os.path.isdir(_p):
            shutil.rmtree(_p, ignore_errors=True)
            print(f"removed {_p}", flush=True)
    except Exception as _e:
        print(f"cleanup skip {_p}: {_e}", flush=True)
print("model files + caches removed; outputs kept", flush=True)
